In [3]:
import pyodbc
from sqlalchemy import create_engine, text
import urllib

# 1. Driver and Database configurations
driver = "ODBC Driver 17 for SQL Server"
server_name = r"LAPTOP-5CHV2K6U\UMASANKAR"
database_name = "ShopSenseDB"

print(f"Connecting to SQL Server: [{server_name}]...")

# 2. Build secure connection string using Windows Authentication
conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

# 3. Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT DB_NAME() AS current_db;")).fetchone()
    print(f"\n SUCCESS! Connected to database: [{result[0]}] on [{server_name}]!")

Connecting to SQL Server: [LAPTOP-5CHV2K6U\UMASANKAR]...

 SUCCESS! Connected to database: [ShopSenseDB] on [LAPTOP-5CHV2K6U\UMASANKAR]!


In [6]:
import os
import pandas as pd
import gc

RAW_DATA_DIR = os.path.join("..", "data", "raw")
geo_file = os.path.join(RAW_DATA_DIR, "olist_geolocation_dataset.csv")

print("Deduplicating & Aggregating 1,000,163 Geolocation rows...")

# 1. Read CSV
df_raw_geo = pd.read_csv(geo_file)

# 2. Group by zip_code_prefix: average coordinates, take primary city/state
df_clean_geo = df_raw_geo.groupby("geolocation_zip_code_prefix").agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first")
).reset_index()

# Clean city name length
df_clean_geo["geolocation_city"] = df_clean_geo["geolocation_city"].astype(str).str.slice(0, 95)

print(f"Reduced from {len(df_raw_geo):,} duplicate rows -> {len(df_clean_geo):,} unique postal zones!")

# 3. Insert cleanly into SQL Server (takes ~2 seconds!)
df_clean_geo.to_sql(name="geolocation", con=engine, if_exists="append", index=False)
print(" Successfully loaded [geolocation] into SQL Server!")

# 4. Clean RAM
del df_raw_geo
del df_clean_geo
gc.collect()

print("\n" + "=" * 60)
print("ALL 9 TABLES ARE NOW 100% STORED IN MICROSOFT SQL SERVER!")

Deduplicating & Aggregating 1,000,163 Geolocation rows...
Reduced from 1,000,163 duplicate rows -> 19,015 unique postal zones!
 Successfully loaded [geolocation] into SQL Server!

ALL 9 TABLES ARE NOW 100% STORED IN MICROSOFT SQL SERVER!


In [7]:
verification_query = """
SELECT 
    t.name AS TableName,
    p.rows AS TotalRows
FROM sys.tables t
INNER JOIN sys.partitions p ON t.object_id = p.object_id
WHERE p.index_id IN (0, 1)
ORDER BY p.rows DESC;
"""

with engine.connect() as conn:
    result_df = pd.read_sql(verification_query, conn)

print("--- SQL SERVER DATABASE ROW COUNT VERIFICATION ---")
result_df

--- SQL SERVER DATABASE ROW COUNT VERIFICATION ---


,TableName,TotalRows
0,order_items,112650
1,order_payments,103886
2,customers,99441
3,orders,99441
4,order_reviews,99224
5,products,32951
6,geolocation,19015
7,sellers,3095
8,product_category_translation,71
